In [ ]:
!pip install pdfplumber
!pip install transformers
!pip install torch
!pip install openai

You should consider upgrading via the 'C:\Users\ayush\Videos\docu3u\PolarBrief\myvenv\Scripts\python.exe -m pip install --upgrade pip' command.


You should consider upgrading via the 'C:\Users\ayush\Videos\docu3u\PolarBrief\myvenv\Scripts\python.exe -m pip install --upgrade pip' command.


You should consider upgrading via the 'C:\Users\ayush\Videos\docu3u\PolarBrief\myvenv\Scripts\python.exe -m pip install --upgrade pip' command.


You should consider upgrading via the 'C:\Users\ayush\Videos\docu3u\PolarBrief\myvenv\Scripts\python.exe -m pip install --upgrade pip' command.


In [8]:
import pdfplumber
import json
from tqdm import tqdm

# Load PDF
pdf_path = "data.pdf"
output = []

with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in tqdm(enumerate(pdf.pages, start=1), desc="Processing pages"):
        text = page.extract_text()
        if not text:
            continue

        lines = text.split('\n')
        for line_num, line in enumerate(lines, start=1):
            cleaned_line = line.strip()
            if cleaned_line:
                output.append({
                    "page": page_num,
                    "line": line_num,
                    "text": cleaned_line
                })
print(output)

with open("amicus_brief_text_with_metadata.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)


Processing pages: 26it [00:02,  8.76it/s]

[{'page': 1, 'line': 1, 'text': 'Case 2:22-cv-00223-Z Document 100 Filed 02/13/23 Page 1 of 26 PageID 3760'}, {'page': 1, 'line': 2, 'text': 'UNITED STATES DISTRICT COURT'}, {'page': 1, 'line': 3, 'text': 'NORTHERN DISTRICT OF TEXAS'}, {'page': 1, 'line': 4, 'text': 'AMARILLO DIVISION'}, {'page': 1, 'line': 5, 'text': 'ALLIANCE FOR HIPPOCRATIC'}, {'page': 1, 'line': 6, 'text': 'MEDICINE, et al.,'}, {'page': 1, 'line': 7, 'text': 'Plaintiffs,'}, {'page': 1, 'line': 8, 'text': 'v. Case No. 2:22-cv-00223-Z'}, {'page': 1, 'line': 9, 'text': 'U.S. FOOD AND DRUG'}, {'page': 1, 'line': 10, 'text': 'ADMINISTRATION, et al.,'}, {'page': 1, 'line': 11, 'text': 'Defendants.'}, {'page': 1, 'line': 12, 'text': 'AMICUS CURIAE BRIEF OF MISSISSIPPI, ALABAMA, ALASKA,'}, {'page': 1, 'line': 13, 'text': 'ARKANSAS, FLORIDA, GEORGIA, IDAHO, INDIANA, IOWA, KANSAS,'}, {'page': 1, 'line': 14, 'text': 'KENTUCKY, LOUISIANA, MONTANA, NEBRASKA, OHIO, OKLAHOMA,'}, {'page': 1, 'line': 15, 'text': 'SOUTH CAROLINA, SO

In [9]:
from typing import List, Dict

def chunk_lines_into_paragraphs(lines: List[Dict], max_gap: int = 1) -> List[Dict]:
    paragraphs = []
    current_para = {
        "page": None,
        "start_line": None,
        "end_line": None,
        "text": []
    }

    for i, line in enumerate(lines):
        content = line["text"].strip()
        if not content:
            continue  # skip blank lines

        # If starting a new paragraph
        if current_para["text"] == []:
            current_para["page"] = line["page"]
            current_para["start_line"] = line["line"]

        # Append line
        current_para["text"].append(content)
        current_para["end_line"] = line["line"]

        # Check next line (if exists)
        if i + 1 < len(lines):
            next_line = lines[i + 1]
            same_page = next_line["page"] == line["page"]
            line_gap = next_line["line"] - line["line"]
            
            if not same_page or line_gap > max_gap:
                # Commit current paragraph
                paragraphs.append({
                    "page": current_para["page"],
                    "start_line": current_para["start_line"],
                    "end_line": current_para["end_line"],
                    "text": " ".join(current_para["text"])
                })
                current_para = {"page": None, "start_line": None, "end_line": None, "text": []}

    # Catch the last paragraph
    if current_para["text"]:
        paragraphs.append({
            "page": current_para["page"],
            "start_line": current_para["start_line"],
            "end_line": current_para["end_line"],
            "text": " ".join(current_para["text"])
        })

    return paragraphs


In [10]:
paragraphs = chunk_lines_into_paragraphs(output)
print (paragraphs)
with open("amicus_paragraph_chunks.json", "w", encoding="utf-8") as f:
    json.dump(paragraphs, f, indent=2, ensure_ascii=False)
 

[{'page': 1, 'start_line': 1, 'end_line': 17, 'text': 'Case 2:22-cv-00223-Z Document 100 Filed 02/13/23 Page 1 of 26 PageID 3760 UNITED STATES DISTRICT COURT NORTHERN DISTRICT OF TEXAS AMARILLO DIVISION ALLIANCE FOR HIPPOCRATIC MEDICINE, et al., Plaintiffs, v. Case No. 2:22-cv-00223-Z U.S. FOOD AND DRUG ADMINISTRATION, et al., Defendants. AMICUS CURIAE BRIEF OF MISSISSIPPI, ALABAMA, ALASKA, ARKANSAS, FLORIDA, GEORGIA, IDAHO, INDIANA, IOWA, KANSAS, KENTUCKY, LOUISIANA, MONTANA, NEBRASKA, OHIO, OKLAHOMA, SOUTH CAROLINA, SOUTH DAKOTA, TENNESSEE, TEXAS, UTAH, AND WYOMING IN SUPPORT OF PLAINTIFFS’ MOTION FOR PRELIMINARY INJUNCTION'}, {'page': 2, 'start_line': 1, 'end_line': 19, 'text': 'Case 2:22-cv-00223-Z Document 100 Filed 02/13/23 Page 2 of 26 PageID 3761 TABLE OF CONTENTS Page TABLE OF AUTHORITIES .......................................................................................... ii INTRODUCTION, INTEREST OF AMICI CURIAE, AND SUMMARY OF ARGUMENT .................................

In [ ]:

import os
from openai import OpenAI

os.environ["GROQ_API_KEY"] = "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"

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def get_argument_analysis(paragraph: str):
    prompt = f"""
You are a legal assistant AI.

Given the paragraph below from a legal brief:
1. Does it contain a legal argument? (yes/no)
2. If yes, summarize it in ≤75 words.
3. Classify it as Pro (supports Plaintiffs) or Con (supports Defendants).
4. Score the argument on a scale of 0 - 1000 , based on legal clarity, relevance, and reasoning quality and other aspects.

Respond in JSON like this:
{{
  "contains_argument": "...",
  "summary": "...",
  "polarity": "Pro/Con",
  "start line": "..",
  "end line":"...",
  "score":"..." 
}}

Paragraph:
\"\"\"{paragraph}\"\"\"
"""

    response = client.chat.completions.create(
        model="llama3-8b-8192", 
        messages=[
            {"role": "system", "content": "You are a legal assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=512
    )

    return response.choices[0].message.content.strip()

# Example
paragraph = """The FDA exceeded its authority by removing critical safety regulations without proper review, putting public health at risk."""
print(get_argument_analysis(paragraph))

{
  "contains_argument": "yes",
  "summary": "The FDA overstepped its authority by removing safety regulations without proper review, putting public health at risk.",
  "polarity": "Pro",
  "start line": "1",
  "end line": "1",
  "score": "800"
}


In [3]:
import json

with open("amicus_paragraph_chunks.json", "r", encoding="utf-8") as f:
    paragraphs = json.load(f)

argument_results = []

for para in paragraphs:
    result = get_argument_analysis(para)
    argument_results.append(result)

# Save the results

print (argument_results)

['{\n  "contains_argument": "no",\n  "summary": "",\n  "polarity": "",\n  "start line": "1",\n  "end line": "17",\n  "score": "0"\n}', '{\n  "contains_argument": "yes",\n  "summary": "The argument states that the FDA\'s actions on mifepristone defy federal law, undermine states\' ability to protect their citizens, and force states to divert resources to investigating and prosecuting violations of their laws, thus supporting injunctive relief.",\n  "polarity": "Pro",\n  "start line": "7",\n  "end line": "14",\n  "score": "800"\n}', '{\n  "contains_argument": "no",\n  "summary": "",\n  "polarity": "",\n  "start line": "1",\n  "end line": "29",\n  "score": "0"\n}', '{\n  "contains_argument": "no",\n  "summary": "",\n  "polarity": "",\n  "start line": "1",\n  "end line": "26",\n  "score": "0"\n}', '{\n  "contains_argument": "no",\n  "summary": "",\n  "polarity": "",\n  "start line": "1",\n  "end line": "32",\n  "score": "0"\n}', '{\n  "contains_argument": "no",\n  "summary": "",\n  "polari

In [ ]:
parsed_results = []

for i, entry in enumerate(argument_results):
    try:
        # Cleanup (some responses contain extra text)
        cleaned = entry.strip()
        if "{" not in cleaned:
            continue  # skip if no JSON structure present

        json_str = cleaned[cleaned.index("{"):]  # grab from first {
        json_str = json_str[:json_str.rindex("}")+1]  # safely end at last }

        data = json.loads(json_str)

        # Only include meaningful arguments
        if data.get("contains_argument", "").strip().lower() == "yes":
            parsed_results.append({
                "index": i,
                "summary": data.get("summary", "").strip(),
                "polarity": data.get("polarity", "").strip(),
                "score": float(data.get("score", "0")),
                "start_line": int(data.get("start line", "0")),
                "end_line": int(data.get("end line", "0"))
            })

    except Exception as e:
        print(f"Error at index {i}: {e}")

# Write to JSON file
with open("arguments_with_polarity.json", "w", encoding="utf-8") as f:
    json.dump(parsed_results, f, indent=4)


In [ ]:
pro_args = [r for r in parsed_results if r["polarity"].lower() == "pro"]
con_args = [r for r in parsed_results if r["polarity"].lower() == "con"]
                                 
# Sort if needed (e.g., by length of summary = heuristic for "strength")
pro_args = sorted(
    [arg for arg in parsed_results if arg["polarity"].lower() == "pro"],
    key=lambda x: x["score"],
    reverse=True
)[:5]

con_args = sorted(
    [arg for arg in parsed_results if arg["polarity"].lower() == "con"],
    key=lambda x: x["score"],
    reverse=True
)[:5]
# Combine for export
top_10 = {
    "top_5_pro": pro_args,
    "top_5_con": con_args
}

# Save to JSON file
with open("top_10_arguments.json", "w") as f:
    json.dump(top_10, f, indent=2)

# Print nicely
from pprint import pprint
pprint(top_10)

{'top_5_con': [{'end_line': 25,
                'index': 19,
                'polarity': 'Con',
                'score': 850.0,
                'start_line': 21,
                'summary': 'The FDA lacks authority to establish a mail-order '
                           'abortion regime and its actions undermine state '
                           'laws and the public interest, as states have the '
                           'authority to balance competing interests and have '
                           'enacted laws reflecting the views of their '
                           'citizens.'},
               {'end_line': 25,
                'index': 7,
                'polarity': 'Con',
                'score': 800.0,
                'start_line': 1,
                'summary': 'The Biden Administration has attacked the '
                           'democratic process and undermined the judgments of '
                           'elected representatives by promoting access to '
                 